## Overlap Control for Training and Test Data ##

In [2]:
from time import time

def iter_kmers(seq, k):
    """Yield unique k-mers from seq in first-seen order."""
    seen = set()
    for start in range(len(seq) - k + 1):
        kmer = seq[start:start + k]
        if kmer not in seen:
            seen.add(kmer)
            yield kmer

def unique_column_indices_by_kmer_cutoff(
    sequences_A,
    sequences_B,
    cutoff_len=9,
    output_file=None,
    buffer_size=100000,
    sep="\t",
    progress_every=None,
    log_func=print,
):
    """Return B indices that share no cutoff_len k-mer with any A sequence.

    LCS(seq_A, seq_B) >= cutoff_len is equivalent to the two sequences sharing
    at least one exact k-mer of length cutoff_len.
    """
    t0 = time()
    sequences_A = list(sequences_A)
    sequences_B = list(sequences_B)
    buffer_size = max(int(buffer_size), 1)

    if cutoff_len <= 0:
        raise ValueError("cutoff_len must be positive")

    if progress_every:
        log_func(f"[Unique-kmer] indexing A k-mers with k={cutoff_len}")

    kmer_to_idx_A = {}
    indexed_sequences = 0
    for i, seq_A in enumerate(sequences_A):
        if len(seq_A) >= cutoff_len:
            indexed_sequences += 1
            for kmer in iter_kmers(seq_A, cutoff_len):
                kmer_to_idx_A.setdefault(kmer, i)

        if progress_every and (i + 1) % progress_every == 0:
            elapsed_min = (time() - t0) / 60
            log_func(
                f"[Unique-kmer] indexed A rows {i + 1}/{len(sequences_A)}, "
                f"unique_kmers {len(kmer_to_idx_A)}, elapsed {elapsed_min:.2f} min"
            )

    unique_indices = []
    n_matches = 0
    line_buffer = [sep.join(["idx_A", "idx_B", "lcs_length"]) + "\n"]
    f = open(output_file, "w", buffering=1024 * 1024) if output_file else None

    try:
        for j, seq_B in enumerate(sequences_B):
            matched_idx_A = None
            if len(seq_B) >= cutoff_len:
                for kmer in iter_kmers(seq_B, cutoff_len):
                    matched_idx_A = kmer_to_idx_A.get(kmer)
                    if matched_idx_A is not None:
                        break

            if matched_idx_A is None:
                unique_indices.append(j)
            else:
                n_matches += 1
                if f is not None:
                    line_buffer.append(sep.join(map(str, [matched_idx_A, j, cutoff_len])) + "\n")
                    if len(line_buffer) >= buffer_size:
                        f.writelines(line_buffer)
                        line_buffer.clear()

            if progress_every and (j + 1) % progress_every == 0:
                elapsed_min = (time() - t0) / 60
                log_func(
                    f"[Unique-kmer] columns {j + 1}/{len(sequences_B)}, "
                    f"matched {n_matches}, unique {len(unique_indices)}, "
                    f"elapsed {elapsed_min:.2f} min"
                )

        if f is not None and line_buffer:
            f.writelines(line_buffer)
    finally:
        if f is not None:
            f.close()

    if progress_every:
        elapsed_min = (time() - t0) / 60
        log_func(
            f"[Unique-kmer] done A {len(sequences_A)}, indexed_A {indexed_sequences}, "
            f"unique_kmers {len(kmer_to_idx_A)}, B {len(sequences_B)}, "
            f"matched {n_matches}, unique {len(unique_indices)}, elapsed {elapsed_min:.2f} min"
        )

    return unique_indices

## SA De-duplication ##

In [ ]:
import pandas as pd
from pathlib import Path
from time import time

PAIR_FILE = "./SA_test.tsv"
UNIQUE_FILE = "./SA_unique_peptides.txt"


def log(message):
    print(message, flush=True)


t0 = time()
log("[SA] Loading PM2023 positive peptides")
data_list = []
for i in range(5):
    data = pd.read_csv(f"./5cv/PM2023/panmix_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    data_list.append(data)
data_all = pd.concat(data_list)
pos_data = data_all[data_all["label"] == 1].copy()
PM_pos_pep = pos_data["peptide"].drop_duplicates()
log(f"[SA] PM positive unique peptides: {len(PM_pos_pep)}")

## SA 测试 ##
log("[SA] Loading SA2023 positive peptides")
SA_data = pd.read_csv("./indep/SA2023.txt", sep=" ", names=["peptide","label","allele","context", "flag"])
SA_pos = SA_data[SA_data["label"] == 1].copy()
SA_pos_pep = SA_pos["peptide"].drop_duplicates()
log(f"[SA] SA positive unique peptides: {len(SA_pos_pep)}")

# PM_pos_pep = PM_pos_pep[:1000]
# SA_pos_pep = SA_pos_pep[:500]

total_pairs = len(PM_pos_pep) * len(SA_pos_pep)
log(f"[SA] Finding unique SA columns by k-mer cutoff: {len(PM_pos_pep)} x {len(SA_pos_pep)} <= {total_pairs}")
unique_col_indices = unique_column_indices_by_kmer_cutoff(
    PM_pos_pep,
    SA_pos_pep,
    cutoff_len=9,
    output_file=str(PAIR_FILE),
    buffer_size=100000,
    progress_every=50000,
    log_func=log,
)
log(f"[SA] Saved first matching pairs to {PAIR_FILE}")
unique_peptides = SA_pos_pep.iloc[unique_col_indices]
unique_peptides.to_csv(UNIQUE_FILE, sep=",", header=False, index=False)
log(f"[SA] Unique SA peptides: {len(unique_peptides)}")
log(f"[SA] Saved unique peptides to {UNIQUE_FILE}")
log(f"[SA] Total elapsed: {(time() - t0) / 60:.2f} min")

[SA] Loading PM2023 positive peptides
[SA] PM positive unique peptides: 436407
[SA] Loading SA2023 positive peptides
[SA] SA positive unique peptides: 125638
[SA] Finding unique SA columns by k-mer cutoff: 436407 x 125638 <= 54829302666
[Unique-kmer] indexing A k-mers with k=9
[Unique-kmer] indexed A rows 50000/436407, unique_kmers 115311, elapsed 0.00 min
[Unique-kmer] indexed A rows 100000/436407, unique_kmers 293918, elapsed 0.00 min
[Unique-kmer] indexed A rows 150000/436407, unique_kmers 412460, elapsed 0.01 min
[Unique-kmer] indexed A rows 200000/436407, unique_kmers 502309, elapsed 0.01 min
[Unique-kmer] indexed A rows 250000/436407, unique_kmers 592501, elapsed 0.01 min
[Unique-kmer] indexed A rows 300000/436407, unique_kmers 671009, elapsed 0.01 min
[Unique-kmer] indexed A rows 350000/436407, unique_kmers 745452, elapsed 0.01 min
[Unique-kmer] indexed A rows 400000/436407, unique_kmers 820805, elapsed 0.02 min
[Unique-kmer] columns 50000/125638, matched 37042, unique 12958, el

In [ ]:
import pandas as pd
# SA2023 is organized as positive blocks followed by matched negative blocks.
# Within each negative block, the 5 negatives for each positive sample are consecutive.
SA_data = pd.read_csv(
    "./indep/SA2023.txt",
    sep=r"\s+",
    names=["peptide", "label", "allele", "context"],
)
sa_pep = pd.read_csv(
    "./SA_unique_peptides.txt",
    header=None,
    names=["peptide"],
)

SA_pos_data = SA_data[SA_data["label"] == 1].copy()
SA_pos_redun = SA_pos_data[SA_pos_data["peptide"].isin(sa_pep["peptide"])]


def label_runs(df):
    labels = df["label"].to_numpy()
    runs = []
    start = 0
    for i in range(1, len(labels)):
        if labels[i] != labels[start]:
            runs.append((start, i, labels[start]))
            start = i
    runs.append((start, len(labels), labels[start]))
    return runs


runs = label_runs(SA_data)
if len(runs) % 2 != 0:
    raise ValueError(f"Expected positive/negative run pairs, got {len(runs)} runs")

block_layout = []
for block_id in range(len(runs) // 2):
    pos_start, pos_end, pos_label = runs[2 * block_id]
    neg_start, neg_end, neg_label = runs[2 * block_id + 1]
    n_pos = pos_end - pos_start
    n_neg = neg_end - neg_start
    if pos_label != 1 or neg_label != 0:
        raise ValueError(f"Unexpected label order in block {block_id}: {pos_label}, {neg_label}")
    if n_neg % n_pos != 0:
        raise ValueError(f"Block {block_id} negative block is not a multiple of positive block")
    block_layout.append(
        {
            "block": block_id,
            "pos_start": pos_start,
            "pos_end": pos_end,
            "neg_start": neg_start,
            "neg_end": neg_end,
            "n_pos": n_pos,
            "n_neg": n_neg,
            "neg_per_pos": n_neg // n_pos,
        }
    )


def paired_negative_indices(pos_indices, layout):
    neg_indices = []
    source_pos_indices = []
    negative_copy_ids = []
    layout_iter = iter(layout)
    current = next(layout_iter)

    for pos_idx in sorted(pos_indices):
        while pos_idx >= current["pos_end"]:
            current = next(layout_iter)
        if pos_idx < current["pos_start"]:
            raise ValueError(f"Positive index {pos_idx} is outside detected positive blocks")

        local_idx = pos_idx - current["pos_start"]
        for k in range(current["neg_per_pos"]):
            neg_indices.append(current["neg_start"] + local_idx * current["neg_per_pos"] + k)
            source_pos_indices.append(pos_idx)
            negative_copy_ids.append(k)

    return neg_indices, source_pos_indices, negative_copy_ids


sa_neg_indices, source_pos_indices, negative_copy_ids = paired_negative_indices(
    SA_pos_redun.index,
    block_layout,
)
SA_neg_redun = SA_data.loc[sa_neg_indices].copy()
SA_neg_redun["source_pos_index"] = source_pos_indices
SA_neg_redun["negative_copy_id"] = negative_copy_ids

SA_redun = pd.concat([SA_pos_redun, SA_neg_redun])
if not Path("./indep_redundancy").exists():
    Path("./indep_redundancy").mkdir(parents=True, exist_ok=True)
SA_redun[["peptide", "label", "allele", "context"]].to_csv(
"./indep_redundancy/SA2023.txt", sep=" ", index=False, header=False)

sa_layout = pd.DataFrame(block_layout)
sa_extract_summary = {
    "n_rows": len(SA_data),
    "n_positive": int((SA_data["label"] == 1).sum()),
    "n_negative": int((SA_data["label"] == 0).sum()),
    "n_sa_peptide": len(sa_pep),
    "n_SA_pos_redun": len(SA_pos_redun),
    "n_SA_neg_redun": len(SA_neg_redun),
    "negative_per_positive": sorted(sa_layout["neg_per_pos"].unique().tolist()),
}
sa_extract_summary


{'n_rows': 1211136,
 'n_positive': 201856,
 'n_negative': 1009280,
 'n_sa_peptide': 35020,
 'n_SA_pos_redun': 47780,
 'n_SA_neg_redun': 238900,
 'negative_per_positive': [5]}

## MA De-duplication ##

In [13]:
import pandas as pd
from pathlib import Path
from time import time

PAIR_FILE = "./MA_test.tsv"
UNIQUE_FILE = "./MA_unique_peptides.txt"


def log(message):
    print(message, flush=True)


t0 = time()
log("[MA] Loading PMC2023 positive peptides")
data_list = []
for i in range(5):
    data = pd.read_csv(f"./5cv/PMC2023/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    data_list.append(data)
data_all = pd.concat(data_list)
pos_data = data_all[data_all["label"] == 1].copy()
PMC_pos_pep = pos_data["peptide"].drop_duplicates()
log(f"[MA] PMC positive unique peptides: {len(PMC_pos_pep)}")

# MA测试 #
log("[MA] Loading MA2024 positive peptides")
MA_data = pd.read_csv("./indep/MA2024.txt", sep="\t", names=["peptide","label","allele","context"])
MA_pos = MA_data[MA_data["label"] == 1].copy()
MA_pos_pep = MA_pos["peptide"].drop_duplicates()
log(f"[MA] MA positive unique peptides: {len(MA_pos_pep)}")

# PMC_pos_pep = PMC_pos_pep[:1000]
# MA_pos_pep = MA_pos_pep[:500]

total_pairs = len(PMC_pos_pep) * len(MA_pos_pep)
log(f"[MA] Finding unique MA columns by k-mer cutoff: {len(PMC_pos_pep)} x {len(MA_pos_pep)} <= {total_pairs}")
unique_col_indices = unique_column_indices_by_kmer_cutoff(
    PMC_pos_pep,
    MA_pos_pep,
    cutoff_len=9,
    output_file=str(PAIR_FILE),
    buffer_size=100000,
    progress_every=10000,
    log_func=log,
)
log(f"[MA] Saved first matching pairs to {PAIR_FILE}")
unique_peptides = MA_pos_pep.iloc[unique_col_indices]
unique_peptides.to_csv(UNIQUE_FILE, sep=",", header=False, index=False)
log(f"[MA] Unique MA peptides: {len(unique_peptides)}")
log(f"[MA] Saved unique peptides to {UNIQUE_FILE}")
log(f"[MA] Total elapsed: {(time() - t0) / 60:.2f} min")


[MA] Loading PMC2023 positive peptides
[MA] PMC positive unique peptides: 531263
[MA] Loading MA2024 positive peptides
[MA] MA positive unique peptides: 146056
[MA] Finding unique MA columns by k-mer cutoff: 531263 x 146056 <= 77594148728
[Unique-kmer] indexing A k-mers with k=9
[Unique-kmer] indexed A rows 10000/531263, unique_kmers 25330, elapsed 0.00 min
[Unique-kmer] indexed A rows 20000/531263, unique_kmers 49292, elapsed 0.00 min
[Unique-kmer] indexed A rows 30000/531263, unique_kmers 72361, elapsed 0.00 min
[Unique-kmer] indexed A rows 40000/531263, unique_kmers 94005, elapsed 0.00 min
[Unique-kmer] indexed A rows 50000/531263, unique_kmers 115311, elapsed 0.00 min
[Unique-kmer] indexed A rows 60000/531263, unique_kmers 144139, elapsed 0.00 min
[Unique-kmer] indexed A rows 70000/531263, unique_kmers 191311, elapsed 0.00 min
[Unique-kmer] indexed A rows 80000/531263, unique_kmers 241785, elapsed 0.00 min
[Unique-kmer] indexed A rows 90000/531263, unique_kmers 290164, elapsed 0.00

In [14]:
# MA2024 uses the same positive-block followed by consecutive matched-negative-block layout.
MA_data = pd.read_csv(
    "./indep/MA2024.txt",
    sep="\t",
    names=["peptide", "label", "allele", "context"],
)
ma_pep = pd.read_csv(
    "./MA_unique_peptides.txt",
    header=None,
    names=["peptide"],
)

MA_pos_data = MA_data[MA_data["label"] == 1].copy()
MA_pos_redun = MA_pos_data[MA_pos_data["peptide"].isin(ma_pep["peptide"])]

def label_runs(df):
    labels = df["label"].to_numpy()
    runs = []
    start = 0
    for i in range(1, len(labels)):
        if labels[i] != labels[start]:
            runs.append((start, i, labels[start]))
            start = i
    runs.append((start, len(labels), labels[start]))
    return runs

ma_runs = label_runs(MA_data)
if len(ma_runs) % 2 != 0:
    raise ValueError(f"Expected positive/negative run pairs, got {len(ma_runs)} runs")

ma_block_layout = []
for block_id in range(len(ma_runs) // 2):
    pos_start, pos_end, pos_label = ma_runs[2 * block_id]
    neg_start, neg_end, neg_label = ma_runs[2 * block_id + 1]
    n_pos = pos_end - pos_start
    n_neg = neg_end - neg_start
    if pos_label != 1 or neg_label != 0:
        raise ValueError(f"Unexpected label order in MA block {block_id}: {pos_label}, {neg_label}")
    if n_neg % n_pos != 0:
        raise ValueError(f"MA block {block_id} negative block is not a multiple of positive block")
    ma_block_layout.append(
        {
            "block": block_id,
            "pos_start": pos_start,
            "pos_end": pos_end,
            "neg_start": neg_start,
            "neg_end": neg_end,
            "n_pos": n_pos,
            "n_neg": n_neg,
            "neg_per_pos": n_neg // n_pos,
        }
    )

def paired_negative_indices(pos_indices, layout):
    neg_indices = []
    source_pos_indices = []
    negative_copy_ids = []
    layout_iter = iter(layout)
    current = next(layout_iter)

    for pos_idx in sorted(pos_indices):
        while pos_idx >= current["pos_end"]:
            current = next(layout_iter)
        if pos_idx < current["pos_start"]:
            raise ValueError(f"Positive index {pos_idx} is outside detected positive blocks")

        local_idx = pos_idx - current["pos_start"]
        for k in range(current["neg_per_pos"]):
            neg_indices.append(current["neg_start"] + local_idx * current["neg_per_pos"] + k)
            source_pos_indices.append(pos_idx)
            negative_copy_ids.append(k)

    return neg_indices, source_pos_indices, negative_copy_ids

ma_neg_indices, source_pos_indices, negative_copy_ids = paired_negative_indices(
    MA_pos_redun.index,
    ma_block_layout,
)
MA_neg_redun = MA_data.loc[ma_neg_indices].copy()
MA_neg_redun["source_pos_index"] = source_pos_indices
MA_neg_redun["negative_copy_id"] = negative_copy_ids

if not Path("./indep_redundancy").exists():
    Path("./indep_redundancy").mkdir(parents=True, exist_ok=True)
MA_redun = pd.concat([MA_pos_redun, MA_neg_redun])
MA_redun = MA_redun[["peptide", "label", "allele", "context"]]
MA_redun[MA_redun["allele"].isin(["A549", "HEK293T"])].to_csv(
    "./indep_redundancy/MA2024.txt",
    sep=" ",
    index=False,
    header=False,
)

ma_layout = pd.DataFrame(ma_block_layout)
ma_extract_summary = {
    "n_rows": len(MA_data),
    "n_positive": int((MA_data["label"] == 1).sum()),
    "n_negative": int((MA_data["label"] == 0).sum()),
    "n_ma_peptide": len(ma_pep),
    "n_MA_pos_redun": len(MA_pos_redun),
    "n_MA_neg_redun": len(MA_neg_redun),
    "n_MA_redun": len(MA_redun),
    "negative_per_positive": sorted(ma_layout["neg_per_pos"].unique().tolist()),
    "output_file": "./indep_redundancy/MA2024.txt",
}
ma_extract_summary


{'n_rows': 1624566,
 'n_positive': 270761,
 'n_negative': 1353805,
 'n_ma_peptide': 15692,
 'n_MA_pos_redun': 19201,
 'n_MA_neg_redun': 96005,
 'n_MA_redun': 115206,
 'negative_per_positive': [5],
 'output_file': './indep_redundancy/MA2024.txt'}

## 5CV De-duplication ##
#### the combinded 5cv dataset construction based on NetMHCIIPan origin folds using 9-mer greedy, load-banlancing strategy ####

In [21]:
import pandas as pd

# def split_interleaved_chunks(data, chunk_size=5):
#     midpoint = len(data) // 2
#     halves = (data.iloc[:midpoint], data.iloc[midpoint:])
#     chunks = []
#     max_length = max(map(len, halves))
#     for start in range(0, max_length, chunk_size):
#         for half in halves:
#             chunk = half.iloc[start:start + chunk_size]
#             if not chunk.empty:
#                 chunks.append(chunk)
#     return pd.concat(chunks, ignore_index=True).copy()

NEG_PER_POS = 10
pos_data_list, neg_data_list = [], []
pan_pos_data, pan_neg_data = [], []
for i in range(5):
    data = pd.read_csv(f"./5cv/PMC2023/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    pan_data = data[data["flag"] == "pan"].copy()
    pan_pos_data.append(pan_data[pan_data["label"] == 1].copy())
    pan_neg_data.append(pan_data[pan_data["label"] == 0].copy())
    cap_data = data[data["flag"] == "captan"].copy()
    cap_pos = cap_data[cap_data["label"] == 1]
    cap_neg = cap_data[cap_data["label"] == 0].iloc[:len(cap_pos) * NEG_PER_POS]
    # cap_neg = split_interleaved_chunks(cap_neg)
    mix_data = data[data["flag"] == "mix"].copy()
    mix_pos = mix_data[mix_data["label"] == 1]
    mix_neg = mix_data[mix_data["label"] == 0].iloc[:len(mix_pos) * NEG_PER_POS]
    # mix_neg = split_interleaved_chunks(mix_neg)
    other_pos = pd.concat([cap_pos, mix_pos])
    other_neg = pd.concat([cap_neg, mix_neg])
    pos_data_list.append(other_pos)
    neg_data_list.append(other_neg)

other_pos_data = pd.concat(pos_data_list)
other_neg_data = pd.concat(neg_data_list)

### Based on peptide identify ###

In [45]:
def pairwise_common_counts(pos_data_list):
    pep_sets = [
        set(df["peptide"].drop_duplicates())
        for df in pos_data_list
    ]

    rows = []
    for i in range(len(pep_sets)):
        for j in range(i + 1, len(pep_sets)):
            common = len(pep_sets[i] & pep_sets[j])
            min_size = min(len(pep_sets[i]), len(pep_sets[j]))
            union_size = len(pep_sets[i] | pep_sets[j])
            rows.append(
                {
                    "fold_i": i,
                    "fold_j": j,
                    "n_i": len(pep_sets[i]),
                    "n_j": len(pep_sets[j]),
                    "common": common,
                    "common_ratio_min": common / min_size if min_size else 0,
                    "common_ratio_union": common / union_size if union_size else 0,
                }
            )
    return pd.DataFrame(rows)


NEG_PER_POS = 10
if len(other_neg_data) != len(other_pos_data) * NEG_PER_POS:
    raise ValueError(
        f"Expected {NEG_PER_POS} negatives per positive, got "
        f"{len(other_neg_data)} negatives for {len(other_pos_data)} positives"
    )

before_common = pairwise_common_counts(pan_pos_data)

pan_pep_sets = [
    set(pan_pos_data[i]["peptide"].drop_duplicates())
    for i in range(5)
]
other_pos_with_pair_id = other_pos_data.copy()
other_pos_with_pair_id["_pair_id"] = range(len(other_pos_with_pair_id))
other_groups = list(other_pos_with_pair_id.groupby("peptide", sort=False))

other_pos_split_parts = [[] for _ in range(5)]
other_neg_split_parts = [[] for _ in range(5)]
other_pos_split_sizes = [0] * 5
other_neg_split_sizes = [0] * 5
other_pos_split_n_peptides = [0] * 5

for peptide, group in other_groups:
    existed_folds = [
        i for i in range(5)
        if peptide in pan_pep_sets[i]
    ]

    if existed_folds:
        target = min(existed_folds, key=lambda i: other_pos_split_sizes[i])
    else:
        target = min(range(5), key=lambda i: other_pos_split_sizes[i])

    pair_ids = group["_pair_id"].to_list()
    neg_indices = []
    for pair_id in pair_ids:
        neg_start = pair_id * NEG_PER_POS
        neg_indices.extend(range(neg_start, neg_start + NEG_PER_POS))

    other_pos_split_parts[target].append(group.drop(columns="_pair_id"))
    other_neg_split_parts[target].append(other_neg_data.iloc[neg_indices])
    other_pos_split_sizes[target] += len(group)
    other_neg_split_sizes[target] += len(neg_indices)
    other_pos_split_n_peptides[target] += 1

other_pos_split = [
    pd.concat(parts) if parts else other_pos_data.iloc[0:0].copy()
    for parts in other_pos_split_parts
]
other_neg_split = [
    pd.concat(parts) if parts else other_neg_data.iloc[0:0].copy()
    for parts in other_neg_split_parts
]

merged_pan_pos_data = [
    pd.concat([pan_pos_data[i], other_pos_split[i]], ignore_index=True)
    for i in range(5)
]
merged_pan_neg_data = [
    pd.concat([pan_neg_data[i], other_neg_split[i]], ignore_index=True)
    for i in range(5)
]

after_common = pairwise_common_counts(merged_pan_pos_data)
overlap_change = before_common.merge(
    after_common,
    on=["fold_i", "fold_j"],
    suffixes=("_before", "_after"),
)
overlap_change["common_delta"] = overlap_change["common_after"] - overlap_change["common_before"]
overlap_change["common_ratio_min_delta"] = (
    overlap_change["common_ratio_min_after"] - overlap_change["common_ratio_min_before"]
)

distribution_summary = pd.DataFrame(
    {
        "fold": range(5),
        "other_pos_rows": other_pos_split_sizes,
        "other_neg_rows": other_neg_split_sizes,
        "neg_per_pos": [
            other_neg_split_sizes[i] / other_pos_split_sizes[i]
            if other_pos_split_sizes[i]
            else 0
            for i in range(5)
        ],
        "other_pos_unique_peptides": other_pos_split_n_peptides,
        "pan_pos_rows_before": [len(df) for df in pan_pos_data],
        "pan_pos_rows_after": [len(df) for df in merged_pan_pos_data],
        "pan_neg_rows_before": [len(df) for df in pan_neg_data],
        "pan_neg_rows_after": [len(df) for df in merged_pan_neg_data],
        "pan_pos_unique_before": [df["peptide"].nunique() for df in pan_pos_data],
        "pan_pos_unique_after": [df["peptide"].nunique() for df in merged_pan_pos_data],
    }
)

pan_pos_pep_0 = merged_pan_pos_data[0]["peptide"].drop_duplicates()
pan_pos_pep_1 = merged_pan_pos_data[1]["peptide"].drop_duplicates()
pan_pos_common_peptides = sorted(set(pan_pos_pep_0) & set(pan_pos_pep_1))
pan_pos_common_count = len(pan_pos_common_peptides)

{
    "n_other_pos_rows": len(other_pos_data),
    "n_other_neg_rows": len(other_neg_data),
    "n_other_pos_unique_peptides": other_pos_data["peptide"].nunique(),
    "after_pan_pos_0_unique": len(pan_pos_pep_0),
    "after_pan_pos_1_unique": len(pan_pos_pep_1),
    "after_pan_0_1_common_peptides": pan_pos_common_count,
    "max_common_delta": int(overlap_change["common_delta"].max()),
    "max_common_ratio_min_delta": float(overlap_change["common_ratio_min_delta"].max()),
    "neg_per_pos_by_fold": distribution_summary["neg_per_pos"].tolist(),
}


{'n_other_pos_rows': 601929,
 'n_other_neg_rows': 6019290,
 'n_other_pos_unique_peptides': 399302,
 'after_pan_pos_0_unique': 107067,
 'after_pan_pos_1_unique': 107771,
 'after_pan_0_1_common_peptides': 0,
 'max_common_delta': 0,
 'max_common_ratio_min_delta': 0.0,
 'neg_per_pos_by_fold': [10.0, 10.0, 10.0, 10.0, 10.0]}

In [46]:
from pathlib import Path

out_dir = Path("./5cv/PMC2023_redundancy")
out_dir.mkdir(parents=True, exist_ok=True)

for i, (pos_df, neg_df) in enumerate(zip(merged_pan_pos_data, merged_pan_neg_data)):
    data_df = pd.concat([pos_df, neg_df], ignore_index=True)

    data_df.to_csv(
        out_dir / f"merge_cv{i}_el",
        sep=" ",
        index=False,
        header=False,
    )

    print(i, len(pos_df), len(neg_df), len(neg_df) / len(pos_df))

0 370052 3804192 10.28015522142834
1 371079 3791959 10.218737788988328
2 379877 3783887 9.960821529073884
3 380036 3782240 9.95232030649728
4 381659 3792373 9.93654807039792


### Based on 9mer overlap ###

In [20]:
# Reassign other positives by shared 9-mer similarity rather than exact peptide identity.
KMER_SIZE = 9
N_FOLDS = len(pan_pos_data)


def peptide_kmer_set(peptide, k=KMER_SIZE):
    peptide = str(peptide).upper()
    return {
        peptide[start : start + k]
        for start in range(len(peptide) - k + 1)
    }


def dataframe_kmer_set(df, k=KMER_SIZE):
    kmers = set()
    for peptide in df["peptide"].dropna().drop_duplicates():
        kmers.update(peptide_kmer_set(peptide, k=k))
    return kmers


if len(other_neg_data) != len(other_pos_data) * NEG_PER_POS:
    raise ValueError(
        f"Expected {NEG_PER_POS} negatives per positive, got "
        f"{len(other_neg_data)} negatives for {len(other_pos_data)} positives"
    )
if other_pos_data["peptide"].isna().any():
    raise ValueError("other_pos_data contains missing peptide values")

# These sets are updated after every assignment, so later peptides can match
# both the original pan data and previously assigned other peptides.
fold_9mer_sets = [dataframe_kmer_set(df) for df in pan_pos_data]

other_pos_with_pair_id = other_pos_data.copy()
other_pos_with_pair_id["_pair_id"] = range(len(other_pos_with_pair_id))
other_groups = list(other_pos_with_pair_id.groupby("peptide", sort=False))

other_pos_split_parts = [[] for _ in range(N_FOLDS)]
other_neg_split_parts = [[] for _ in range(N_FOLDS)]
other_pos_split_sizes = [0] * N_FOLDS
other_neg_split_sizes = [0] * N_FOLDS
other_pos_split_n_peptides = [0] * N_FOLDS
assignment_rows = []

for peptide, group in other_groups:
    peptide_9mers = peptide_kmer_set(peptide)
    common_9mer_counts = [
        len(peptide_9mers & fold_9mers)
        for fold_9mers in fold_9mer_sets
    ]
    similarities = [
        common / len(peptide_9mers) if peptide_9mers else 0.0
        for common in common_9mer_counts
    ]
    best_similarity = max(similarities)

    if best_similarity > 0:
        candidate_folds = [
            i for i, similarity in enumerate(similarities)
            if similarity == best_similarity
        ]
    else:
        candidate_folds = list(range(N_FOLDS))

    target = min(
        candidate_folds,
        key=lambda i: (other_pos_split_sizes[i], i),
    )

    pair_ids = group["_pair_id"].to_list()
    neg_indices = [
        neg_index
        for pair_id in pair_ids
        for neg_index in range(
            pair_id * NEG_PER_POS,
            (pair_id + 1) * NEG_PER_POS,
        )
    ]

    other_pos_split_parts[target].append(group.drop(columns="_pair_id"))
    other_neg_split_parts[target].append(other_neg_data.iloc[neg_indices])
    other_pos_split_sizes[target] += len(group)
    other_neg_split_sizes[target] += len(neg_indices)
    other_pos_split_n_peptides[target] += 1
    fold_9mer_sets[target].update(peptide_9mers)

    assignment_rows.append(
        {
            "peptide": peptide,
            "target_fold": target,
            "n_rows": len(group),
            "n_9mers": len(peptide_9mers),
            "common_9mers": common_9mer_counts[target],
            "similarity": similarities[target],
        }
    )

other_pos_split = [
    pd.concat(parts, ignore_index=True)
    if parts else other_pos_data.iloc[0:0].copy()
    for parts in other_pos_split_parts
]
other_neg_split = [
    pd.concat(parts, ignore_index=True)
    if parts else other_neg_data.iloc[0:0].copy()
    for parts in other_neg_split_parts
]

merged_pan_pos_data = [
    pd.concat([pan_pos_data[i], other_pos_split[i]], ignore_index=True)
    for i in range(N_FOLDS)
]
merged_pan_neg_data = [
    pd.concat([pan_neg_data[i], other_neg_split[i]], ignore_index=True)
    for i in range(N_FOLDS)
]

assignment_details = pd.DataFrame(assignment_rows)
distribution_summary_9mer = pd.DataFrame(
    {
        "fold": range(N_FOLDS),
        "other_pos_rows": other_pos_split_sizes,
        "other_neg_rows": other_neg_split_sizes,
        "other_pos_unique_peptides": other_pos_split_n_peptides,
        "neg_per_pos": [
            other_neg_split_sizes[i] / other_pos_split_sizes[i]
            if other_pos_split_sizes[i] else 0
            for i in range(N_FOLDS)
        ],
        "total_unique_9mers_after": [len(kmers) for kmers in fold_9mer_sets],
    }
)

distribution_summary_9mer


,fold,other_pos_rows,other_neg_rows,other_pos_unique_peptides,neg_per_pos,total_unique_9mers_after
0,0,120387,1203870,80530,10.0,229002
1,1,120382,1203820,81049,10.0,221734
2,2,120387,1203870,79683,10.0,228020
3,3,120392,1203920,80004,10.0,228665
4,4,120381,1203810,78036,10.0,221483


In [ ]:
from pathlib import Path

out_dir = Path("./5cv/PMC2023_9mer_redundancy")
out_dir.mkdir(parents=True, exist_ok=True)

for i, (pos_df, neg_df) in enumerate(zip(merged_pan_pos_data, merged_pan_neg_data)):
    data_df = pd.concat([pos_df, neg_df], ignore_index=True)

    data_df.to_csv(
        out_dir / f"merge_cv{i}_el",
        sep=" ",
        index=False,
        header=False,
    )

    print(i, len(pos_df), len(neg_df), len(neg_df) / len(pos_df))

0 249667 2600342 10.415241101146728
1 250689 2588059 10.323783652254386
2 259492 2580037 9.942645630693818
3 259656 2578440 9.930215361863388
4 261270 2588483 9.907310445133387


## Overlap Calculation ##

In [3]:
def test_peptide_9mer_overlap(train_data, test_data, peptide_col="peptide", k=9):
    """Measure the proportion of unique test peptides sharing at least one k-mer with training peptides."""
    if k <= 0:
        raise ValueError("k must be a positive integer")
    for name, data in (("train_data", train_data), ("test_data", test_data)):
        if peptide_col not in data.columns:
            raise KeyError(f"{name} does not contain column {peptide_col!r}")

    train_peptides = (
        train_data[peptide_col].dropna().astype(str).str.upper().drop_duplicates()
    )
    test_peptides = (
        test_data[peptide_col].dropna().astype(str).str.upper()
    )

    train_kmers = set()
    for peptide in train_peptides:
        train_kmers.update(
            peptide[start : start + k]
            for start in range(max(len(peptide) - k + 1, 1))
        )
    
    overlap_count = sum(
        any(
            peptide[start : start + k] in train_kmers
            for start in range(max(len(peptide) - k + 1, 1))
        )
        for peptide in test_peptides
    )
    test_count = len(test_peptides)
    overlap_ratio = overlap_count / test_count if test_count else 0.0

    result = {
        "k": k,
        "train_unique_peptides": len(train_peptides),
        "test_peptides": test_count,
        "test_peptides_with_overlap": overlap_count,
        "overlap_ratio": overlap_ratio,
    }
    print(
        f"Test peptides with >=1 shared {k}-mer: "
        f"{overlap_count:,}/{test_count:,} ({overlap_ratio:.2%})"
    )
    return result



In [ ]:
from pathlib import Path
import pandas as pd

# test_path = Path("./indep_redundancy/SA2023.txt")
test_path = Path("./indep_redundancy/MA2024.txt")

train_data_list = []
for i in range(5):
    # df_data = pd.read_csv(f"./5cv/Pan2023/c00{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    # df_data = pd.read_csv(f"./5cv/PM2023/panmix_cv{i}_el", sep=r"\s+", names=["peptide","label","allele","context", "flag"])
    # df_data = pd.read_csv(f"./5cv/PMC2023/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    # df_data = pd.read_csv(f"./5cv/PMC2023_redundancy/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    df_data = pd.read_csv(f"./5cv/PMC2023_9mer_redundancy/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    train_data_list.append(df_data)
train_data=pd.concat(train_data_list, ignore_index=True)
test_data = pd.read_csv(test_path, sep=r"\s+", names=["peptide","label","allele","context"])

In [ ]:
train_data = train_data[train_data["label"] == 1].copy()
overlap_stats = test_peptide_9mer_overlap(train_data, test_data[test_data["label"] == 0], k=9)
overlap_stats

Test peptides with >=1 shared 30-mer: 165/238,900 (0.07%)


{'k': 30,
 'train_unique_peptides': 283091,
 'test_peptides': 238900,
 'test_peptides_with_overlap': 165,
 'overlap_ratio': 0.0006906655504395144}

In [40]:
overlap_ratios = []
for i in range(5):
    for j in range(5):
        if i == j:
            continue
        fold1_data = train_data_list[i]
        fold2_data = train_data_list[j]
        overlap_stats = test_peptide_9mer_overlap(fold1_data[fold1_data["label"] == 1], fold2_data[fold2_data["label"] == 1], k=9)
        overlap_ratios.append(overlap_stats["overlap_ratio"])

mean_overlap_ratio = sum(overlap_ratios) / len(overlap_ratios) if overlap_ratios else 0.0
print(f"Mean overlap ratio between folds: {mean_overlap_ratio:.2%}")

Test peptides with >=1 shared 9-mer: 17,932/250,689 (7.15%)
Test peptides with >=1 shared 9-mer: 16,180/259,492 (6.24%)
Test peptides with >=1 shared 9-mer: 15,848/259,656 (6.10%)
Test peptides with >=1 shared 9-mer: 23,828/261,270 (9.12%)
Test peptides with >=1 shared 9-mer: 15,244/249,667 (6.11%)
Test peptides with >=1 shared 9-mer: 17,623/259,492 (6.79%)
Test peptides with >=1 shared 9-mer: 20,694/259,656 (7.97%)
Test peptides with >=1 shared 9-mer: 20,149/261,270 (7.71%)
Test peptides with >=1 shared 9-mer: 16,836/249,667 (6.74%)
Test peptides with >=1 shared 9-mer: 19,842/250,689 (7.91%)
Test peptides with >=1 shared 9-mer: 19,909/259,656 (7.67%)
Test peptides with >=1 shared 9-mer: 22,883/261,270 (8.76%)
Test peptides with >=1 shared 9-mer: 15,179/249,667 (6.08%)
Test peptides with >=1 shared 9-mer: 18,419/250,689 (7.35%)
Test peptides with >=1 shared 9-mer: 16,674/259,492 (6.43%)
Test peptides with >=1 shared 9-mer: 19,497/261,270 (7.46%)
Test peptides with >=1 shared 9-mer: 19,

# Set-level Overlap Calculation #

In [43]:
pan_pos_data, pan_neg_data = [], []
for i in range(5):
    # pan_data = pd.read_csv(f"./5cv/Pan2023/c00{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    # pan_data = pd.read_csv(f"./5cv/PMC2023/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    pan_data = pd.read_csv(f"./5cv/PMC2023_redundancy/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    # pan_data = pd.read_csv(f"./5cv/PMC2023_9mer_redundancy/merge_cv{i}_el", sep=" ", names=["peptide","label","allele","context", "flag"])
    pan_pos_data.append(pan_data[pan_data["label"] == 1].copy())
    pan_neg_data.append(pan_data[pan_data["label"] == 0].copy())
pan_pos_pep_0 = pan_pos_data[0]["peptide"].drop_duplicates()
pan_pos_pep_1 = pan_pos_data[1]["peptide"].drop_duplicates()
pan_pos_common_peptides = sorted(set(pan_pos_pep_0) & set(pan_pos_pep_1))
pan_pos_common_count = len(pan_pos_common_peptides)

In [44]:
def pairwise_common_9mer_counts(pos_data_list, k=9):
    kmer_sets = []
    for df in pos_data_list:
        kmers = set()
        for peptide in df["peptide"].dropna().drop_duplicates():
            peptide = str(peptide).upper()
            kmers.update(
                peptide[start : start + k]
                for start in range(len(peptide) - k + 1)
            )
        kmer_sets.append(kmers)

    rows = []
    for i in range(len(kmer_sets)):
        for j in range(i + 1, len(kmer_sets)):
            common = len(kmer_sets[i] & kmer_sets[j])
            min_size = min(len(kmer_sets[i]), len(kmer_sets[j]))
            union_size = len(kmer_sets[i] | kmer_sets[j])
            rows.append(
                {
                    "fold_i": i,
                    "fold_j": j,
                    "n_i": len(kmer_sets[i]),
                    "n_j": len(kmer_sets[j]),
                    "common": common,
                    "common_ratio_min": common / min_size if min_size else 0,
                    "common_ratio_union": common / union_size if union_size else 0,
                }
            )
    return pd.DataFrame(rows)

ma_path = "./indep_redundancy/MA2024.txt"
ma_data = pd.read_csv(ma_path, sep=r"\s+", names=["peptide","label","allele","context"])
# pan_pos_data.append(ma_data.copy())
pairwise_common_9mer_counts(pan_pos_data)


,fold_i,fold_j,n_i,n_j,common,common_ratio_min,common_ratio_union
0,0,1,386348,386362,150060,0.388406,0.241002
1,0,2,386348,378068,143690,0.380064,0.231487
2,0,3,386348,380392,144729,0.380473,0.232679
3,0,4,386348,373965,144388,0.386100,0.234425
4,1,2,386362,378068,145029,0.383606,0.234144
5,1,3,386362,380392,144717,0.380442,0.232650
6,1,4,386362,373965,141867,0.379359,0.229388
7,2,3,378068,380392,144091,0.381125,0.234535
8,2,4,378068,373965,138171,0.369476,0.225085
9,3,4,380392,373965,142157,0.380135,0.232207
